# Nextflow Pipeline Resource Analysis

Analyze Nextflow trace files to identify resource optimization opportunities
and compare resource usage across different pipeline runs.
## Workflow
1. **Load & Preprocess** — Import trace files and organize into comparison groups
2. **Optimization Analysis** — Identify processes with poor resource efficiency
3. **Process Consistency** — Filter to processes present across all runs
4. **Resource Analysis** — Compare scaling trends and per-process contributions


### Setup & Configuration

In [9]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display, clear_output
from ipywidgets import Dropdown, interact, FloatSlider, IntSlider

# Add project root to path
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "workflow_helpers.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from workflow_helpers import (
    load_comparison_groups,
    identify_consistent_processes,
    create_optimization_bubble_chart,
    RESOURCE_CONFIG,
    create_efficiency_bar,
    create_efficiency_dashboard,
    create_process_composition_plot,
    create_optimization_priority_plot,
)

# Configure matplotlib styling
plt.style.use("seaborn-v0_8-dark")
plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.size": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.15,
        "grid.linestyle": "-",
        "axes.titleweight": "medium",
        "axes.labelweight": "regular",
        "figure.dpi": 120,
    }
)

PLOT_WIDTH = 800
PLOT_HEIGHT = 400

## 1. Load & Preprocess Trace Files
Organize trace files into comparison groups. Each group should contain runs
from the same pipeline that you want to compare.

In [10]:
# ── Configuration ──────────────────────────────────────────────────────────────
# Define comparison groups of trace files
comparison_groups = {
    "rnaseq": {
        "trace_files": [
            Path("../data/rnaseq_runs/rnaseq_test_trace_2026-03-19_16-41-59.txt"),
            Path("../data/rnaseq_runs/rnaseq_test_full_trace_2026-03-19_16-51-08.txt"),
        ],
        "labels": ["rnaseq test", "rnaseq test full"],  # or None for auto-labels
    },
}

# ── Load all trace files ───────────────────────────────────────────────────────
run_summary_df, process_df, trace_files, labels = load_comparison_groups(
    comparison_groups
)

Loading 1 comparison group(s):
  • rnaseq: 2 trace file(s)

Loading rnaseq:
  • rnaseq_test_trace_2026-03-19_16-41-59.txt
    → 219 completed tasks
  • rnaseq_test_full_trace_2026-03-19_16-51-08.txt
    → 303 completed tasks
⚠️  WARNING: 27 tasks used more CPU cores than requested!
   Scheduler may not be enforcing CPU limits properly.
   Examples (showing up to 5):
     NFCORE_RNASEQ:RNASEQ:BEDTOOLS_GENOMECOV_REV: requested 1.0 cores, used 1.0 (102.3%)
     NFCORE_RNASEQ:RNASEQ:BEDTOOLS_GENOMECOV_FW: requested 1.0 cores, used 1.0 (102.3%)
     NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:BAM_SORT_STATS_SAMTOOLS:BAM_STATS_SAMTOOLS:SAMTOOLS_STATS: requested 1.0 cores, used 1.3 (130.5%)
     NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:BAM_SORT_STATS_SAMTOOLS:BAM_STATS_SAMTOOLS:SAMTOOLS_STATS: requested 1.0 cores, used 1.2 (124.9%)
     NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:BAM_SORT_STATS_SAMTOOLS:BAM_STATS_SAMTOOLS:SAMTOOLS_STATS: requested 1.0 cores, used 1.2 (122.2%)


✓ Ready: 2 run(s) across 1 group(s)


## 2. Optimization Analysis

Identify processes with poor CPU and memory efficiency. Bubble size reflects
computational burden (runtime × task count); color shows overall efficiency.
Top-right processes are highest-priority optimization targets.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# -----------------------------
# Controls
# -----------------------------
run_label_widget = widgets.Dropdown(
    options=labels,
    description="Run:",
    style={"description_width": "initial"},
)

top_n_widget = widgets.IntSlider(
    value=20,
    min=5,
    max=30,
    step=1,
    description="Top N:",
    style={"description_width": "initial"},
    continuous_update=False,
)

# Impact sliders
impact_cpu_widget = widgets.FloatSlider(
    value=0.4,
    min=0.0,
    max=1.0,
    step=0.05,
    description="CPU cost",
    style={"description_width": "initial"},
    continuous_update=False,
)
impact_mem_widget = widgets.FloatSlider(
    value=0.3,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Memory cost",
    style={"description_width": "initial"},
    continuous_update=False,
)
impact_time_widget = widgets.FloatSlider(
    value=0.3,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Time cost",
    style={"description_width": "initial"},
    continuous_update=False,
)

# Inefficiency sliders
ineff_cpu_widget = widgets.FloatSlider(
    value=0.4,
    min=0.0,
    max=1.0,
    step=0.05,
    description="CPU ineff.",
    style={"description_width": "initial"},
    continuous_update=False,
)
ineff_mem_widget = widgets.FloatSlider(
    value=0.3,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Memory ineff.",
    style={"description_width": "initial"},
    continuous_update=False,
)
ineff_time_widget = widgets.FloatSlider(
    value=0.3,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Time ineff.",
    style={"description_width": "initial"},
    continuous_update=False,
)

# Variability sliders
var_runtime_widget = widgets.FloatSlider(
    value=0.6,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Runtime CV",
    style={"description_width": "initial"},
    continuous_update=False,
)
var_cpu_widget = widgets.FloatSlider(
    value=0.2,
    min=0.0,
    max=1.0,
    step=0.05,
    description="CPU CV",
    style={"description_width": "initial"},
    continuous_update=False,
)
var_mem_widget = widgets.FloatSlider(
    value=0.2,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Memory CV",
    style={"description_width": "initial"},
    continuous_update=False,
)

# -----------------------------
# Plot function
# -----------------------------
output = widgets.Output()


def show_optimization_dashboard(
    run_label,
    top_n,
    impact_cpu,
    impact_mem,
    impact_time,
    ineff_cpu,
    ineff_mem,
    ineff_time,
    var_runtime,
    var_cpu,
    var_mem,
):
    idx = labels.index(run_label)
    stem = trace_files[idx].stem
    metrics = process_df[process_df["filename"] == stem].copy()

    impact_weights = {
        "cpu_cost": impact_cpu,
        "mem_cost": impact_mem,
        "time_cost": impact_time,
    }
    inefficiency_weights = {
        "cpu_ineff": ineff_cpu,
        "mem_ineff": ineff_mem,
        "time_ineff": ineff_time,
    }
    variability_weights = {
        "runtime_cv": var_runtime,
        "cpu_cv": var_cpu,
        "mem_cv": var_mem,
    }

    fig = create_optimization_priority_plot(
        metrics=metrics,
        impact_weights=impact_weights,
        inefficiency_weights=inefficiency_weights,
        variability_weights=variability_weights,
        top_n=top_n,
    )

    with output:
        clear_output(wait=True)
        fig.show()


# -----------------------------
# Grouped layout in two rows
# -----------------------------
general_box = widgets.VBox(
    [
        widgets.HTML("<h4 style='margin:0;'>General</h4>"),
        widgets.HTML(
            "<div style='font-size:12px;color:#666;'>"
            "Weights are normalized within each section."
            "</div>"
        ),
        run_label_widget,
        top_n_widget,
    ],
    layout=widgets.Layout(
        border="1px solid #ddd",
        padding="10px 12px",
        margin="6px 0 10px 0",
        width="100%",
    ),
)

shared_box_layout = widgets.Layout(
    border="1px solid #ddd",
    padding="10px 12px",
    margin="0 8px 0 0",
    width="33%",
)

impact_box = widgets.VBox(
    [
        widgets.HTML(
            "<h4 style='margin:0;'>Impact</h4>"
            "<div style='font-size:12px;color:#666;'>"
            "Relative weighting of compute cost components"
            "</div>"
        ),
        impact_cpu_widget,
        impact_mem_widget,
        impact_time_widget,
    ],
    layout=shared_box_layout,
)

ineff_box = widgets.VBox(
    [
        widgets.HTML(
            "<h4 style='margin:0;'>Inefficiency</h4>"
            "<div style='font-size:12px;color:#666;'>"
            "Relative weighting of resource inefficiencies"
            "</div>"
        ),
        ineff_cpu_widget,
        ineff_mem_widget,
        ineff_time_widget,
    ],
    layout=shared_box_layout,
)

var_box = widgets.VBox(
    [
        widgets.HTML(
            "<h4 style='margin:0;'>Variability</h4>"
            "<div style='font-size:12px;color:#666;'>"
            "Relative weighting of input-dependent variability"
            "</div>"
        ),
        var_runtime_widget,
        var_cpu_widget,
        var_mem_widget,
    ],
    layout=widgets.Layout(
        border="1px solid #ddd",
        padding="10px 12px",
        margin="0",
        width="33%",
    ),
)

slider_row = widgets.HBox(
    [impact_box, ineff_box, var_box],
    layout=widgets.Layout(
        width="100%",
        align_items="flex-start",
        justify_content="space-between",
    ),
)

controls = widgets.VBox(
    [
        widgets.HTML(
            "<h3 style='margin:0 0 8px 0;'>Optimization Priority Dashboard</h3>"
        ),
        general_box,
        slider_row,
    ]
)

# -----------------------------
# Bind widgets to function
# -----------------------------
interactive = widgets.interactive_output(
    show_optimization_dashboard,
    {
        "run_label": run_label_widget,
        "top_n": top_n_widget,
        "impact_cpu": impact_cpu_widget,
        "impact_mem": impact_mem_widget,
        "impact_time": impact_time_widget,
        "ineff_cpu": ineff_cpu_widget,
        "ineff_mem": ineff_mem_widget,
        "ineff_time": ineff_time_widget,
        "var_runtime": var_runtime_widget,
        "var_cpu": var_cpu_widget,
        "var_mem": var_mem_widget,
    },
)

display(controls, output, interactive)

Output()

Output()

In [12]:
def show_efficiency_dashboard(run_label, sort_by):
    """Show efficiency chart for a given run with chosen sorting."""
    idx = labels.index(run_label)
    stem = trace_files[idx].stem
    metrics = process_df[(process_df["filename"] == stem)].copy()

    fig = create_efficiency_bar(metrics, sort_by=sort_by)
    fig.update_layout(width=PLOT_WIDTH, height=PLOT_HEIGHT)  # Set width/height here
    display(fig)


interact(
    show_efficiency_dashboard,
    run_label=Dropdown(options=labels, description="Run:"),
    sort_by=Dropdown(
        options=[
            ("Avg CPU Efficiency", "avg_cpu_efficiency"),
            ("Avg Memory Efficiency", "avg_mem_efficiency"),
            ("Avg Time Efficiency", "avg_time_efficiency"),
            ("Sum of Efficiencies", "efficiency_sum"),
            ("Total Runtime", "total_runtime_hours"),
        ],
        description="Sort by:",
    ),
)

interactive(children=(Dropdown(description='Run:', options=('rnaseq test', 'rnaseq test full'), value='rnaseq …

<function __main__.show_efficiency_dashboard(run_label, sort_by)>

## Step 3 — Process Consistency Check

Identify processes that do not appear in every run and exclude them. This ensures all comparisons are made on a consistent set of processes.


In [13]:
# Mark consistent processes per group
process_df["is_consistent"] = False

for group_name, group_config in comparison_groups.items():
    group_trace_files = group_config["trace_files"]
    group_labels_local = group_config.get("labels")

    # Use filenames if no labels provided
    if not group_labels_local or len(group_labels_local) != len(group_trace_files):
        group_labels_local = [f.stem for f in group_trace_files]

    group_filenames = [f.stem for f in group_trace_files]
    group_process_df = process_df[process_df["group"] == group_name]

    if len(group_trace_files) == 1:
        print(f"Group: {group_name}")
        print(
            f"  Single run — all {len(group_process_df['process'].unique())} "
            "processes marked as consistent."
        )
        process_df.loc[process_df["group"] == group_name, "is_consistent"] = True
        print()
    else:
        print(f"Group: {group_name}")
        print(
            f"  Comparing {len(group_trace_files)} runs: {', '.join(group_labels_local)}"
        )

        consistent_procs = identify_consistent_processes(
            group_process_df, group_filenames, group_labels_local, verbose=True
        )

        mask = (process_df["group"] == group_name) & (
            process_df["process"].isin(consistent_procs)
        )
        process_df.loc[mask, "is_consistent"] = True
        print()

print(
    f"\n✓ Process consistency analysis complete for "
    f"{len(comparison_groups)} group(s)."
)

Group: rnaseq
  Comparing 2 runs: rnaseq test, rnaseq test full
  EXCLUDED — processes not present in all 2 run(s):
──────────────────────────────────────────────────────────────────────
  • NFCORE_RNASEQ:PREPARE_GENOME:BBMAP_BBSPLIT
    Found in: rnaseq test
  • NFCORE_RNASEQ:PREPARE_GENOME:CUSTOM_CATADDITIONALFASTA
    Found in: rnaseq test
  • NFCORE_RNASEQ:PREPARE_GENOME:GTF2BED
    Found in: rnaseq test
  • NFCORE_RNASEQ:PREPARE_GENOME:GUNZIP_ADDITIONAL_FASTA
    Found in: rnaseq test
  • NFCORE_RNASEQ:PREPARE_GENOME:GUNZIP_GTF
    Found in: rnaseq test
  • NFCORE_RNASEQ:PREPARE_GENOME:MAKE_TRANSCRIPTS_FASTA
    Found in: rnaseq test full
  • NFCORE_RNASEQ:PREPARE_GENOME:SALMON_INDEX
    Found in: rnaseq test full
  • NFCORE_RNASEQ:PREPARE_GENOME:STAR_GENOMEGENERATE
    Found in: rnaseq test
  • NFCORE_RNASEQ:PREPARE_GENOME:UNTAR_SALMON_INDEX
    Found in: rnaseq test
  • NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:STAR_ALIGN
    Found in: rnaseq test
  • NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:STAR_

## 4. Resource Analysis


### 4.2 Per-Process Resource Composition

Compare CPU, memory, and I/O contributions across top processes for each run.

In [14]:
def show_process_composition_dashboard(run_label, sort_by):
    idx = labels.index(run_label)
    stem = trace_files[idx].stem

    pp = process_df[process_df["filename"] == stem].copy()

    fig = create_process_composition_plot(pp, run_label, sort_by)
    fig.update_layout(width=PLOT_WIDTH, height=PLOT_HEIGHT)  # Set width/height here

    display(fig)


interact(
    show_process_composition_dashboard,
    run_label=Dropdown(options=labels, description="Run:"),
    sort_by=Dropdown(
        options=[
            ("CPU", "cpu_h"),
            ("Memory", "memory_h"),
            ("I/O", "io_h"),
        ],
        description="Sort by:",
    ),
)

interactive(children=(Dropdown(description='Run:', options=('rnaseq test', 'rnaseq test full'), value='rnaseq …

<function __main__.show_process_composition_dashboard(run_label, sort_by)>

In [15]:
TOP_N_PROCESSES = 15


def plot_process_composition(run_label, sort_by):
    """Display per-process resource bar chart."""
    idx = labels.index(run_label)
    pp = process_df[process_df["filename"] == trace_files[idx].stem].copy()
    pp = pp.nlargest(TOP_N_PROCESSES, sort_by).sort_values(sort_by, ascending=False)

    x = np.arange(len(pp))
    width = 0.25

    figsize = (max(10, len(pp) * 0.45), 6)
    fig, ax = plt.subplots(figsize=figsize)

    for i, resource in enumerate(["CPU", "Memory", "IO"]):
        cfg = RESOURCE_CONFIG[resource]
        ax.bar(
            x + (i - 1) * width,
            pp[f"{resource.lower()}_h_mean"].values,
            width,
            color=cfg["color"],
            alpha=0.9,
            label=cfg["ylabel"],
        )

    ax.set_xticks(x)
    ax.set_xticklabels(
        pp["process_display"].values, fontsize=9, rotation=45, ha="right"
    )
    ax.set_ylabel("Resource Contribution (hours)", fontweight="bold")
    ax.set_title(
        f"Per-Process Resource Contribution — {run_label} (sorted by {sort_by})",
        fontsize=13,
        pad=10,
    )
    ax.legend(loc="upper right", framealpha=0.95, fontsize=10)
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()


sort_options = [
    ("CPU", "cpu_h_mean"),
    ("Memory", "memory_h_mean"),
    ("I/O", "io_h_mean"),
]

interact(
    plot_process_composition,
    run_label=Dropdown(options=labels, description="Run:"),
    sort_by=Dropdown(options=sort_options, description="Sort by:"),
)

interactive(children=(Dropdown(description='Run:', options=('rnaseq test', 'rnaseq test full'), value='rnaseq …

<function __main__.plot_process_composition(run_label, sort_by)>

### 4.3 Process Scaling Explorer (Interactive)

Scaling curves per process for the top contributors per resource type.
Use buttons to switch between CPU, Memory, and I/O views.
Click processes in the legend to show/hide them.

In [16]:
TOP_PROCESSES_PER_RESOURCE = 15


def create_scaling_explorer(group_name):
    """Create interactive process scaling explorer for a specific comparison group."""
    # Filter to only consistent processes of this group
    group_process_df = process_df[
        (process_df["group"] == group_name) & (process_df["is_consistent"])
    ].copy()

    if group_process_df.empty:
        print(f"No consistent processes found for {group_name}")
        return

    # Get the run labels for this group
    group_run_df = run_summary_df[run_summary_df["group"] == group_name]
    group_labels_list = group_run_df["label"].tolist()

    # Add converted columns
    long_df = group_process_df.copy()
    long_df["run_label"] = long_df["label"]

    run_order = {lbl: i for i, lbl in enumerate(group_labels_list)}
    long_df["run_idx"] = long_df["run_label"].map(run_order)

    # Identify top processes per resource
    top_cpu = (
        long_df.groupby("process")["cpu_h"]
        .sum()
        .nlargest(TOP_PROCESSES_PER_RESOURCE)
        .index.tolist()
    )
    top_mem = (
        long_df.groupby("process")["memory_h"]
        .sum()
        .nlargest(TOP_PROCESSES_PER_RESOURCE)
        .index.tolist()
    )
    top_io = (
        long_df.groupby("process")["io_h"]
        .sum()
        .nlargest(TOP_PROCESSES_PER_RESOURCE)
        .index.tolist()
    )

    # Assign stable colors
    import plotly.colors

    palette = plotly.colors.qualitative.Dark24 + plotly.colors.qualitative.Light24
    all_procs = list(dict.fromkeys(top_cpu + top_mem + top_io))
    proc_color = {p: palette[i % len(palette)] for i, p in enumerate(all_procs)}

    # Build display name lookup
    proc_display = {}
    for proc in all_procs:
        row = long_df[long_df["process"] == proc]
        if not row.empty:
            proc_display[proc] = row["process_display"].iloc[0]

    resource_config = [
        ("CPU", top_cpu, "cpu_h", "CPU contribution (hours)"),
        ("Memory", top_mem, "memory_h", "Memory contribution (hours)"),
        ("IO", top_io, "io_h", "I/O contribution (hours)"),
    ]

    fig = go.Figure()
    trace_ranges = {}

    for resource, procs, metric, _ in resource_config:
        start = len(fig.data)
        is_first_resource = resource == "CPU"
        legend_shown_for_resource = set()

        for proc in procs:
            subset = long_df[long_df["process"] == proc].sort_values("run_idx")
            if subset.empty:
                continue

            display_name = proc_display.get(proc, proc)
            color = proc_color.get(proc, "#888888")

            show_in_legend = display_name not in legend_shown_for_resource
            if show_in_legend:
                legend_shown_for_resource.add(display_name)

            customdata = [
                [row.cpu_h, row.memory_h, row.io_h, row.run_label]
                for row in subset.itertuples()
            ]

            rc = RESOURCE_CONFIG["CPU"]["color"]
            mc = RESOURCE_CONFIG["Memory"]["color"]
            ic = RESOURCE_CONFIG["IO"]["color"]

            hovertemplate = (
                f"<b>{display_name}</b><br>"
                "<i>%{customdata[3]}</i><br><br>"
                f"<b style='color:{rc}'>CPU</b>      %{{customdata[0]:.3f}} h<br>"
                f"<b style='color:{mc}'>Memory</b>   %{{customdata[1]:.3f}} h<br>"
                f"<b style='color:{ic}'>IO</b>      %{{customdata[2]:.3f}} h<br>"
                "<extra></extra>"
            )

            fig.add_trace(
                go.Scatter(
                    x=subset["run_label"].tolist(),
                    y=subset[metric].tolist(),
                    mode="lines+markers",
                    name=display_name,
                    legendgroup=f"{resource}_{display_name}",
                    showlegend=show_in_legend,
                    customdata=customdata,
                    hovertemplate=hovertemplate,
                    marker=dict(
                        size=10,
                        color=color,
                        line=dict(width=2, color="white"),
                        symbol="circle",
                    ),
                    line=dict(width=2.5, color=color),
                    visible=is_first_resource,
                )
            )

        trace_ranges[resource] = (start, len(fig.data))

    total_traces = len(fig.data)

    def vis_list(active_resource):
        result = [False] * total_traces
        s, e = trace_ranges[active_resource]
        for i in range(s, e):
            result[i] = True
        return result

    buttons = [
        dict(
            label=f" {res} ",
            method="update",
            args=[
                {"visible": vis_list(res)},
                {
                    "yaxis.title.text": ytitle,
                    "title.text": f"Process Scaling Explorer — {group_name} — Top {TOP_PROCESSES_PER_RESOURCE} by {res}",
                },
            ],
        )
        for res, _, _, ytitle in resource_config
    ]

    fig.update_layout(
        title=dict(
            text=f"Process Scaling Explorer — {group_name} — Top {TOP_PROCESSES_PER_RESOURCE} by CPU",
            font=dict(size=16, color="#1a1a2e", family="sans-serif"),
            x=0,
            xanchor="left",
            pad=dict(l=4, b=10),
        ),
        updatemenus=[
            dict(
                type="buttons",
                direction="right",
                x=0.0,
                xanchor="left",
                y=1.0,
                yanchor="bottom",
                pad=dict(r=6, t=6, b=6),
                buttons=buttons,
                showactive=True,
                bgcolor="#f2f4f8",
                bordercolor="#c8cdd6",
                borderwidth=1,
                font=dict(size=12, color="#222"),
                active=0,
            )
        ],
        xaxis=dict(
            title=dict(text="Pipeline Run", font=dict(size=12, color="#333")),
            tickfont=dict(size=11, color="#333"),
            showgrid=False,
            linecolor="#c8cdd6",
            linewidth=1,
            showline=True,
            ticks="outside",
            ticklen=5,
        ),
        yaxis=dict(
            title=dict(
                text="CPU contribution (hours)", font=dict(size=12, color="#333")
            ),
            tickfont=dict(size=10, color="#333"),
            gridcolor="#e4e9f0",
            gridwidth=1,
            zeroline=False,
            linecolor="#c8cdd6",
            linewidth=1,
            showline=True,
        ),
        hovermode="closest",
        hoverlabel=dict(
            bgcolor="#e8edf3",
            bordercolor="#c8cdd6",
            font=dict(size=11, color="#2d3436"),
            namelength=0,
        ),
        height=580,
        width=1100,
        plot_bgcolor="#f7f9fc",
        paper_bgcolor="white",
        legend=dict(
            title=dict(text="Process", font=dict(size=10, color="#444")),
            font=dict(size=9, color="#222"),
            bgcolor="rgba(255,255,255,0.97)",
            bordercolor="#c8cdd6",
            borderwidth=1,
            groupclick="togglegroup",
            itemsizing="constant",
            tracegroupgap=1,
            x=1.01,
            xanchor="left",
            y=1,
            yanchor="top",
        ),
        margin=dict(t=110, r=240, l=60, b=70),
    )

    fig.show()


if len(comparison_groups) > 1:
    interact(
        create_scaling_explorer,
        group_name=Dropdown(options=comparison_groups.keys(), description="Group:"),
    )
else:
    create_scaling_explorer(list(comparison_groups.keys())[0])